In [37]:
# Import the pandas library for data manipulation.
import json
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import joblib

In [38]:
# Define the path to the dataset and load the CSV file into a pandas DataFrame.
path = '/content/bank-additional-full.csv'
df = pd.read_csv(path, sep=';')

In [8]:
# Display the dimensions (rows, columns) of the DataFrame.
df.shape

(41188, 21)

In [9]:
# Display the first 5 rows of the DataFrame to preview the data.
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [10]:
df[df["pdays"] == 999]["poutcome"].value_counts()

,count
poutcome,
nonexistent,35563
failure,4110


In [11]:
# Target encoding
df['target'] = df['y'].map({'yes': 1, 'no': 0})
if df['target'].isna().any():
    raise ValueError('Unexpected target values in y column.')

# Correct pdays handling: preserve the sentinel information
df['no_previous_contact'] = (df['pdays'] == 999).astype(int)
df['pdays_clean'] = df['pdays'].where(df['pdays'] != 999, -1)

# Drop leakage and redundant/original columns
df = df.drop(columns=['y', 'duration', 'pdays'])

# Do NOT replace 'unknown'. Keep it as category.
print(df[['pdays_clean', 'no_previous_contact', 'target']].head())
print(df['no_previous_contact'].value_counts())


   pdays_clean  no_previous_contact  target
0           -1                    1       0
1           -1                    1       0
2           -1                    1       0
3           -1                    1       0
4           -1                    1       0
no_previous_contact
1    39673
0     1515
Name: count, dtype: int64


In [12]:
print(df['pdays_clean'].value_counts())

pdays_clean
-1     39673
 3       439
 6       412
 4       118
 9        64
 2        61
 7        60
 12       58
 10       52
 5        46
 13       36
 11       28
 1        26
 15       24
 14       20
 8        18
 0        15
 16       11
 17        8
 18        7
 22        3
 19        3
 21        2
 25        1
 26        1
 27        1
 20        1
Name: count, dtype: int64


In [13]:
# Check for missing values in each column after cleaning and dropping specified columns.
df.isna().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


### Imputing Missing Values

I will now impute the missing values in the DataFrame. For categorical columns, I'll use the mode (most frequent value), and for numerical columns, I'll use the median.

In [14]:
print("No missing values to impute at this stage as 'pdays' was transformed and other columns were handled previously.")
print("Current missing values status:")
df.isna().sum()

No missing values to impute at this stage as 'pdays' was transformed and other columns were handled previously.
Current missing values status:


,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [15]:
subset = df[(df["no_previous_contact"] == 1) & (df["poutcome"] != "nonexistent")]
subset

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,target,no_previous_contact,pdays_clean
24013,38,blue-collar,single,unknown,no,yes,no,telephone,oct,tue,...,1,failure,-0.1,93.798,-40.4,4.968,5195.8,0,1,-1
24019,40,services,married,high.school,no,yes,no,telephone,oct,tue,...,1,failure,-0.1,93.798,-40.4,4.968,5195.8,1,1,-1
24076,36,admin.,married,university.degree,no,yes,no,telephone,nov,wed,...,1,failure,-0.1,93.200,-42.0,4.663,5195.8,0,1,-1
24102,36,admin.,married,high.school,no,yes,no,telephone,nov,wed,...,1,failure,-0.1,93.200,-42.0,4.286,5195.8,1,1,-1
24113,29,self-employed,married,university.degree,no,yes,no,telephone,nov,thu,...,1,failure,-0.1,93.200,-42.0,4.245,5195.8,0,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41166,32,admin.,married,university.degree,no,no,no,telephone,nov,wed,...,1,failure,-1.1,94.767,-50.8,1.030,4963.6,1,1,-1
41170,40,management,divorced,university.degree,no,yes,no,cellular,nov,wed,...,4,failure,-1.1,94.767,-50.8,1.030,4963.6,0,1,-1
41173,62,retired,married,university.degree,no,yes,no,cellular,nov,thu,...,2,failure,-1.1,94.767,-50.8,1.031,4963.6,1,1,-1
41175,34,student,single,unknown,no,yes,no,cellular,nov,thu,...,2,failure,-1.1,94.767,-50.8,1.031,4963.6,0,1,-1


In [16]:
subset = df[(df["no_previous_contact"] == 1) & (df["poutcome"] == "success")]
subset

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,target,no_previous_contact,pdays_clean


In [17]:
# List all column names in the DataFrame.
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'target',
 'no_previous_contact',
 'pdays_clean']

In [18]:
# Display a concise summary of the DataFrame, including data types and non-null values.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  41188 non-null  int64  
 1   job                  41188 non-null  object 
 2   marital              41188 non-null  object 
 3   education            41188 non-null  object 
 4   default              41188 non-null  object 
 5   housing              41188 non-null  object 
 6   loan                 41188 non-null  object 
 7   contact              41188 non-null  object 
 8   month                41188 non-null  object 
 9   day_of_week          41188 non-null  object 
 10  campaign             41188 non-null  int64  
 11  previous             41188 non-null  int64  
 12  poutcome             41188 non-null  object 
 13  emp.var.rate         41188 non-null  float64
 14  cons.price.idx       41188 non-null  float64
 15  cons.conf.idx        41188 non-null 

In [19]:
# Check for missing values in each column.
df.isna().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [20]:
# Analyze the distribution and proportion of the 'target' column.
df["target"].value_counts()
df["target"].value_counts(normalize=True)

,proportion
target,
0,0.887346
1,0.112654


In [21]:
# The 'duration' column was removed to prevent data leakage.
# Therefore, analysis on this column cannot be performed.

In [22]:
# Display the top 5 value counts for the 'pdays_clean' column.
df["pdays_clean"].value_counts().head()

,count
pdays_clean,
-1,39673
3,439
6,412
4,118
9,64


In [23]:
# Iterate through object (categorical) columns and display the top 5 value counts for each.
for col in df.select_dtypes(include="object").columns:
    print(col)
    print(df[col].value_counts().head())
    print()

job
job
admin.         10422
blue-collar     9254
technician      6743
services        3969
management      2924
Name: count, dtype: int64

marital
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64

education
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
Name: count, dtype: int64

default
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

housing
housing
yes        21576
no         18622
unknown      990
Name: count, dtype: int64

loan
loan
no         33950
yes         6248
unknown      990
Name: count, dtype: int64

contact
contact
cellular     26144
telephone    15044
Name: count, dtype: int64

month
month
may    13769
jul     7174
aug     6178
jun     5318
nov     4101
Name: count, dtype: int64

day_of_week
day_of_week
thu    8623
mon    8514
wed    8134
tue    8090
fri    7827
Name: co

In [24]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
41183,False
41184,False
41185,False
41186,False


### Duplicate Rows

I will check for and display any duplicate rows in the DataFrame.

In [25]:
df[df.duplicated()]

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,target,no_previous_contact,pdays_clean
10,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,-1
11,25,services,single,high.school,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,-1
16,35,blue-collar,married,basic.6y,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,-1
31,59,technician,married,unknown,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,-1
104,52,admin.,divorced,university.degree,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39985,27,admin.,single,high.school,no,no,no,cellular,jun,tue,...,0,nonexistent,-1.7,94.055,-39.8,0.761,4991.6,1,1,-1
40401,31,student,single,unknown,no,yes,no,cellular,aug,thu,...,0,nonexistent,-1.7,94.027,-38.3,0.904,4991.6,1,1,-1
40404,41,entrepreneur,married,university.degree,no,yes,no,cellular,aug,thu,...,0,nonexistent,-1.7,94.027,-38.3,0.904,4991.6,1,1,-1
40806,35,technician,married,professional.course,no,yes,no,cellular,sep,thu,...,2,failure,-1.1,94.199,-37.5,0.878,4963.6,0,1,-1


This code identifies and displays all rows that are exact duplicates of other rows in the DataFrame. This helps in understanding the extent of data redundancy before removing it.

In [26]:
df = df.drop_duplicates()

After identifying duplicate rows, this step removes them from the DataFrame, ensuring that each row is unique. This is crucial for maintaining data integrity and preventing bias in analysis or model training.

In [27]:
df.duplicated().sum()


np.int64(0)

This code confirms that all duplicate rows have been successfully removed by summing the boolean series returned by `df.duplicated()`. A result of 0 indicates no remaining duplicates.

### Feature Engineering on 'pdays'

For the 'pdays' column (days since last contact):
1.  A new binary feature `no_previous_contact` is created, indicating whether the client was never previously contacted (original `pdays` was 999).
2.  The `pdays` column is cleaned to `pdays_clean` where 999 is replaced with -1 for better numerical representation, as 999 originally signified 'no contact' rather than a large number of days.
3.  The original `pdays` column is dropped to avoid redundancy and potential misinterpretation.

In [28]:
X = df.drop(columns=["target"])
y = df["target"]

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

In [30]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [31]:
X.head(50)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,no_previous_contact,pdays_clean
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
5,45,services,married,basic.9y,unknown,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
7,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
8,24,technician,single,professional.course,no,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1
9,25,services,single,high.school,no,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,1,-1


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np

# Detect binary columns: only 0/1 or True/False values
binary_cols = [
    col for col in X_train.columns
    if X_train[col].dropna().nunique() <= 2
    and set(X_train[col].dropna().unique()).issubset({0, 1, True, False})
]

# Numeric columns excluding binary columns
numeric_cols = [
    col for col in X_train.select_dtypes(include=["number"]).columns.tolist()
    if col not in binary_cols
]

# Categorical columns
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric:", numeric_cols)
print("Binary:", binary_cols)
print("Categorical:", categorical_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_transformer = Pipeline(steps=[
    # Binary columns should stay 0/1, not become 0.5
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_transformer = Pipeline(steps=[
    # This will not touch "unknown", because unknown is not missing.
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("bin", binary_transformer, binary_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

clf.fit(X_train, y_train)

Numeric: ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'pdays_clean']
Binary: ['no_previous_contact']
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'campaign',
                                                   'previous', 'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed',
                                                   'pdays_clean']),
                                                 ('bin',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_freque...
                                                  ['no_previous_contact']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'day_of_week',
                                                   'poutcome'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [33]:
for col in df.select_dtypes(include="int").columns:
    print(col)
    print(df[col].value_counts().head())
    print()

age
age
31    1825
32    1764
33    1741
35    1671
36    1670
Name: count, dtype: int64

campaign
campaign
1    16368
2    10223
3     5230
4     2625
5     1586
Name: count, dtype: int64

previous
previous
0    33858
1     4484
2      752
3      216
4       70
Name: count, dtype: int64

target
target
0    34806
1     4598
Name: count, dtype: int64

no_previous_contact
no_previous_contact
1    37890
0     1514
Name: count, dtype: int64

pdays_clean
pdays_clean
-1    37890
 3      438
 6      412
 4      118
 9       64
Name: count, dtype: int64



In [34]:
val_proba = clf.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.0, 1.0, 1001)
rows = []
for t in thresholds:
    preds = (val_proba >= t).astype(int)
    rows.append({
        'threshold': float(t),
        'recall': recall_score(y_val, preds, zero_division=0),
        'precision': precision_score(y_val, preds, zero_division=0),
        'f1': f1_score(y_val, preds, zero_division=0),
    })

th_df = pd.DataFrame(rows)
valid = th_df[th_df['recall'] >= 0.75]
if valid.empty:
    raise RuntimeError('No threshold achieved recall >= 0.75. Try another classifier or class weighting.')

best_row = valid.sort_values(['threshold', 'f1'], ascending=[False, False]).iloc[0]
operating_threshold = float(best_row['threshold'])
print(best_row)


threshold    0.360000
recall       0.750000
precision    0.224026
f1           0.345000
Name: 360, dtype: float64


In [35]:
test_proba = clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= operating_threshold).astype(int)

metrics = {
    'test_auc': float(roc_auc_score(y_test, test_proba)),
    'test_average_precision': float(average_precision_score(y_test, test_proba)),
    'test_precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'test_recall': float(recall_score(y_test, test_pred, zero_division=0)),
    'test_f1': float(f1_score(y_test, test_pred, zero_division=0)),
    'operating_threshold': operating_threshold,
}

print(json.dumps(metrics, indent=2))
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))

{
  "test_auc": 0.804517856768861,
  "test_average_precision": 0.4706228094686359,
  "test_precision": 0.2296728215095562,
  "test_recall": 0.7714907508161044,
  "test_f1": 0.3539690464303545,
  "operating_threshold": 0.36
}
[[4584 2378]
 [ 210  709]]
              precision    recall  f1-score   support

           0       0.96      0.66      0.78      6962
           1       0.23      0.77      0.35       919

    accuracy                           0.67      7881
   macro avg       0.59      0.71      0.57      7881
weighted avg       0.87      0.67      0.73      7881



In [40]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

models = {
    "logistic": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "rf": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),

    "svc": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

results = []
fitted_models = {}

target_recall = 0.75

for name, model in models.items():
    clf = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    clf.fit(X_train, y_train)
    fitted_models[name] = clf

    probs = clf.predict_proba(X_val)[:, 1]

    threshold_rows = []

    for threshold in np.arange(0.01, 1.00, 0.01):
        preds = (probs >= threshold).astype(int)

        threshold_rows.append({
            "threshold": threshold,
            "precision": precision_score(y_val, preds, zero_division=0),
            "recall": recall_score(y_val, preds, zero_division=0),
            "f1": f1_score(y_val, preds, zero_division=0)
        })

    threshold_df = pd.DataFrame(threshold_rows)

    valid_thresholds = threshold_df[
        threshold_df["recall"] >= target_recall
    ]

    if len(valid_thresholds) > 0:
        best_row = valid_thresholds.sort_values(
            by="threshold",
            ascending=False
        ).iloc[0]

        operating_threshold = best_row["threshold"]
    else:
        operating_threshold = 0.5

    preds = (probs >= operating_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()

    metrics = {
        "model": name,
        "operating_threshold": operating_threshold,
        "roc_auc": roc_auc_score(y_val, probs),
        "avg_precision": average_precision_score(y_val, probs),
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0),
        "f1": f1_score(y_val, preds, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    results.append(metrics)

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["recall", "precision", "roc_auc"],
    ascending=False
).reset_index(drop=True)

results_df

,model,operating_threshold,roc_auc,avg_precision,precision,recall,f1,tn,fp,fn,tp
0,svc,0.05,0.774626,0.380221,0.156456,0.852174,0.264374,2734,4227,136,784
1,gradient_boosting,0.06,0.795327,0.487264,0.199782,0.797826,0.319547,4021,2940,186,734
2,rf,0.05,0.765519,0.426890,0.186975,0.773913,0.301184,3865,3096,208,712
3,logistic,0.36,0.786298,0.465862,0.224026,0.750000,0.345000,4571,2390,230,690
